This notebook answers the research question: which embeddings predict a field's crop better, AEF, OlmoEarth, or both together? It trains the same LightGBM classifier three times on the same fields, changing only the features, and scores each model in two tests. **Within countries:** each country gets its own model, trained on its training blocks and tested on its held-out test blocks. **Across countries:** a model trained on 17 countries is tested on the 18th (leave one country out), for every country in turn.

In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import pyarrow as pa
import lightgbm as lgb
import ultraplot as uplt
from pathlib import Path
from IPython.display import display
from sklearn.metrics import f1_score
pd.set_option('display.max_rows',20)
warnings.filterwarnings('ignore',message="The argument 'eval_set' is deprecated")

In [2]:
INTERIMDIR   = Path('../data/interim')
PROCDIR      = Path('../data/processed')
FIGDIR       = Path('../figs')
COLLECTIONS  = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
TOPCROPS     = 25
FEATURESETS  = {'aef':[f'aef{i:03d}' for i in range(128)],
                'olmoearth':[f'olmoearth{i:03d}' for i in range(128)],
                'both':[f'aef{i:03d}' for i in range(128)]+[f'olmoearth{i:03d}' for i in range(128)]}
TRAINFIELDS  = 50000
VALFIELDS    = 10000
MINCOUNTRIES = 5
EARLYSTOP    = 50
SEED         = 42
NCORES       = len(os.sched_getaffinity(0))
PARAMS       = dict(n_estimators=2000,learning_rate=0.1,num_leaves=31,max_bin=63,colsample_bytree=0.5,class_weight='balanced',
                    metric='multi_error',random_state=SEED,n_jobs=NCORES,verbose=-1)

pa.set_cpu_count(NCORES)
pa.set_io_thread_count(NCORES)

`TRAINFIELDS` and `VALFIELDS` are the fields each model trains and early-stops on, sampled at random from the training and validation blocks. Every model uses the same number, so the three feature sets and the two tests are compared on equal terms. Test fields are never sampled: every field in the test blocks is scored. `TOPCROPS` is how many crops the confusion-matrix figure shows. `MINCOUNTRIES` sets which crops the across-country test uses: a crop must appear in at least this many countries, because a model can only predict crops it has seen in other countries. `PARAMS` are the LightGBM settings. `class_weight='balanced'` makes rare crops count as much as common ones, and early stopping watches the error rate on the validation fields.

In [3]:
def load_collection(collection):
    table  = pd.read_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet")
    blocks = table.chip.str[4:-4]+'_'+(table.chip.str[-4:-2].astype(int)//6).astype(str)+'_'+(table.chip.str[-2:].astype(int)//6).astype(str)
    mixed  = int((table.groupby(blocks).split.nunique()>1).sum())
    sample = pd.concat([table[table.split=='train'].sample(min(TRAINFIELDS,(table.split=='train').sum()),random_state=SEED),
                        table[table.split=='val'].sample(min(VALFIELDS,(table.split=='val').sum()),random_state=SEED),
                        table[table.split=='test']])
    return sample.assign(country=collection.split('/')[1]),{'country':collection.split('/')[1],'blocks':blocks.nunique(),'mixedblocks':mixed}

def save_rows(rows,filename):
    filepath = PROCDIR/filename
    pd.DataFrame(rows).to_csv(filepath,mode='a',header=not filepath.exists(),index=False)

def run_model(experiment,country,featureset,train,val,test):
    start = time.perf_counter()
    cols  = FEATURESETS[featureset]
    val   = val[val.crop.isin(train.crop.unique())]
    model = lgb.LGBMClassifier(**PARAMS).fit(train[cols],train.crop,eval_set=[(val[cols],val.crop)],callbacks=[lgb.early_stopping(EARLYSTOP,verbose=False)])
    pred  = model.predict(test[cols])
    crops = sorted(test.crop.unique())
    f1s   = f1_score(test.crop,pred,labels=crops,average=None,zero_division=0)
    run   = {'experiment':experiment,'country':country,'featureset':featureset}
    save_rows([{**run,'accuracy':(pred==test.crop).mean(),'macrof1':f1s.mean(),'trainfields':len(train),'testfields':len(test),
                'crops':len(crops),'rounds':model.best_iteration_}],'results_metrics.csv')
    save_rows([{**run,'crop':crop,'f1':f1,'support':(test.crop==crop).sum()} for crop,f1 in zip(crops,f1s)],'results_perclass.csv')
    save_rows(pd.crosstab(test.crop.to_numpy(),pred).stack().rename('fields').rename_axis(['true','predicted']).reset_index()
              .query('fields>0').assign(**run).to_dict('records'),'results_confusion.csv')
    minutes,seconds = divmod(round(time.perf_counter()-start),60)
    print(f'- {country} {featureset}: macro F1 {f1s.mean():.3f}, accuracy {(pred==test.crop).mean():.3f} ({minutes}m {seconds}s)')

def finished_runs():
    filepath = PROCDIR/'results_metrics.csv'
    return set(pd.read_csv(filepath)[['experiment','country','featureset']].itertuples(index=False,name=None)) if filepath.exists() else set()

In [4]:
loaded = [load_collection(collection) for collection in COLLECTIONS]
fields = pd.concat([sample for sample,_ in loaded],ignore_index=True)
blocks = pd.DataFrame([check for _,check in loaded])

print(f'Loaded {len(fields):,} fields from {fields.country.nunique()} countries...')
print(f"- {(fields.split=='train').sum():,} training, {(fields.split=='val').sum():,} validation and {(fields.split=='test').sum():,} test fields")
print(f'- {blocks.mixedblocks.sum()} of {blocks.blocks.sum():,} 6 km blocks have fields in more than one split (should be 0)')

Loaded 1,407,922 fields from 18 countries...
- 900,000 training, 180,000 validation and 327,922 test fields
- 0 of 46,165 6 km blocks have fields in more than one split (should be 0)


## Test 1: within countries

One model per country and feature set, trained on that country's training blocks and tested on its test blocks. Finished runs are skipped, so the cell can be rerun after a crash.

In [ ]:
done = finished_runs()
runs = [(country,featureset) for country in fields.country.unique() for featureset in FEATURESETS if ('withincountry',country,featureset) not in done]
print(f'Running {len(runs)} of {fields.country.nunique()*len(FEATURESETS)} within-country models...')

for country,featureset in runs:
    data = fields[fields.country==country]
    run_model('withincountry',country,featureset,data[data.split=='train'],data[data.split=='val'],data[data.split=='test'])

Running 54 of 54 within-country models...


## Test 2: across countries (leave one country out)

For each country, a model is trained on the other 17 and tested on this country's test blocks. Only crops found in at least `MINCOUNTRIES` countries are used, and the training and validation fields are sampled down to `TRAINFIELDS` and `VALFIELDS` so the model is the same size as in test 1.

In [ ]:
shared = fields.groupby('crop').country.nunique().loc[lambda counts:counts>=MINCOUNTRIES].index
pool   = fields[fields.crop.isin(shared)]
print(f'{len(shared)} crops appear in ≥ {MINCOUNTRIES} countries and cover {fields.crop.isin(shared).mean():.1%} of fields')

done = finished_runs()
runs = [(country,featureset) for country in fields.country.unique() for featureset in FEATURESETS if ('acrosscountries',country,featureset) not in done]
print(f'Running {len(runs)} of {fields.country.nunique()*len(FEATURESETS)} leave-one-country-out models...')

for country,featureset in runs:
    others = pool[pool.country!=country]
    train  = others[others.split=='train'].sample(TRAINFIELDS,random_state=SEED)
    val    = others[others.split=='val'].sample(VALFIELDS,random_state=SEED)
    test   = pool[(pool.country==country)&(pool.split=='test')]
    run_model('acrosscountries',country,featureset,train,val,test[test.crop.isin(train.crop.unique())])

## Results

In [ ]:
metrics = pd.read_csv(PROCDIR/'results_metrics.csv').drop_duplicates(['experiment','country','featureset'],keep='last')
summary = metrics.pivot_table(index='experiment',columns='featureset',values=['macrof1','accuracy'],aggfunc='mean')[['macrof1','accuracy']]
wins    = metrics.loc[metrics.groupby(['experiment','country']).macrof1.idxmax()].groupby(['experiment','featureset']).size().unstack(fill_value=0)

print('Mean over countries, by test and feature set...')
display(summary.style.format('{:.3f}'))
print('Number of countries where each feature set has the highest macro F1...')
display(wins)

In [ ]:
percountry = metrics.pivot_table(index='country',columns=['experiment','featureset'],values='macrof1')
percountry = percountry.loc[percountry[('withincountry','both')].sort_values().index]

fig,axs = uplt.subplots(ncols=2,refwidth=3,refheight=0.22*len(percountry),sharey=True)
for ax,experiment,title in zip(axs,['withincountry','acrosscountries'],['Within countries','Across countries (leave one out)']):
    for featureset,color in zip(FEATURESETS,['#2a78d6','#eb6834','#1baf7a']):
        ax.scatter(percountry[(experiment,featureset)],percountry.index,color=color,label=featureset,s=25)
    ax.format(title=title,xlabel='Macro F1',ylabel='',ygrid=True,xlim=(0,1))
axs[0].legend(loc='b',ncols=3,frame=False)
fig.savefig(FIGDIR/'03_macrof1_by_country.png',dpi=200)

In [ ]:
confusion = pd.read_csv(PROCDIR/'results_confusion.csv').drop_duplicates(['experiment','country','featureset','true','predicted'],keep='last')
pairs     = confusion[confusion.true!=confusion.predicted].merge(
            confusion.groupby(['experiment','country','featureset','true']).fields.sum().rename('truefields').reset_index())
pairs     = pairs.assign(share=pairs.fields/pairs.truefields).sort_values('fields',ascending=False)
top5      = pairs.groupby(['experiment','country','featureset']).head(5)
top5.to_csv(PROCDIR/'results_top5confusions.csv',index=False)

print(f"Saved the 5 most confused crop pairs for each of the {top5.groupby(['experiment','country','featureset']).ngroups} runs to {PROCDIR/'results_top5confusions.csv'}")
print('Most confused pairs over all countries, test 1 with both feature sets...')
display(pairs.query("experiment=='withincountry' and featureset=='both'").groupby(['true','predicted']).fields.sum()
        .sort_values(ascending=False).head(10).reset_index())

In [ ]:
fig,axs = uplt.subplots(ncols=3,refwidth=4,share=False)
for ax,featureset in zip(axs,FEATURESETS):
    counts = (confusion.query("experiment=='acrosscountries' and featureset==@featureset")
              .pivot_table(index='true',columns='predicted',values='fields',aggfunc='sum',fill_value=0))
    top    = counts.sum(axis=1).nlargest(TOPCROPS).index
    matrix = counts.reindex(index=top,columns=top,fill_value=0).div(counts.loc[top].sum(axis=1),axis=0)
    ax.imshow(matrix.to_numpy(),cmap='Blues',vmin=0,vmax=1)
    ax.format(title=featureset,xlabel='Predicted crop',ylabel='True crop',grid=False,xticks=range(len(top)),yticks=range(len(top)),
              xticklabels=list(top),yticklabels=list(top) if featureset=='aef' else [],xrotation=90,ticklabelsize=6,xtickminor=False,ytickminor=False)
fig.format(suptitle=f'Across-country confusion: share of each true crop predicted as each crop ({TOPCROPS} most common shared crops)')
fig.savefig(FIGDIR/'03_confusion_acrosscountries.png',dpi=200)

## Data issues

Issues found while building and testing the data, to keep in mind when reading the results. The numbers come from `data/interim/build_log.csv` and the cells above.

1. **Coarse labels in four collections.** Croatia (10 crops), Lithuania (21), Valencia (8) and Castilla-La Mancha (7) label far fewer crops than the others (30–93), so their national crop codes were mapped to broad HCAT classes. Their scores are not comparable to the other countries.
2. **`not_known_and_other` is a class.** It appears in 16 countries and means "unknown or other", not a crop, so errors on it are not real crop confusions.
3. **Many tiny fields in some countries.** Fields under 5 pixels were dropped: 1.5 M in Castilla-La Mancha, 473 k in the Netherlands, 437 k in Valencia and 372 k in Austria. Small fields are under-represented.
4. **Duplicate field ids.** Czechia (7,389) and Catalonia (14,335) have fields sharing an id. They were dropped because their labels are ambiguous.
5. **Fields outside the chips.** 3–20% of fields per country fall outside every 2 km chip, have no split, and were dropped.
6. **OlmoEarth uses half its dimensions.** FTW stores only the first 64 of OlmoEarth's 128 embedding dimensions (AEF has 64 in total), so OlmoEarth may be handicapped.
7. **Across-country coverage.** Crops found in fewer than `MINCOUNTRIES` countries can't be tested across countries. The table below shows the share of each country's test fields that test 2 scores.

In [ ]:
coverage = (metrics.query("experiment=='acrosscountries' and featureset=='both'").set_index('country').testfields
            /fields[fields.split=='test'].groupby('country').size()).rename('Test fields scored in test 2 (%)')*100
display(coverage.sort_values().to_frame().style.format('{:.1f}'))